In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 140.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225

## Dataset

- We Download the Dataset from Tool Ace Hugging Face Repo
- Shuffle and select a 5k Sample

In [2]:
from datasets import load_dataset

# Download ToolACE dataset from Hugging Face
dataset = load_dataset("Team-ACE/ToolACE")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data.json:   0%|          | 0.00/37.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['system', 'conversations'],
        num_rows: 11300
    })
})


In [3]:
# Take 5k Sample from Dataset
toolace_train = dataset["train"]
toolace_5k = toolace_train.shuffle(seed=42).select(range(5000))

print(toolace_5k)
print(toolace_5k[0].keys())

Dataset({
    features: ['system', 'conversations'],
    num_rows: 5000
})
dict_keys(['system', 'conversations'])


## Converting Tool Ace Output to JSON

- Tool Ace has a different output to what our Model needs
- We Convert that output to JSON

In [4]:
import re
import ast
import json


def split_top_level(text, delimiter=","):
    """
    Splits a string by delimiter, but only at top level.
    So commas inside strings, lists, dicts, or parentheses are ignored.
    """
    parts = []
    current = []

    depth = 0
    in_string = False
    quote_char = None
    escape = False

    # for char in text
    for ch in text:
        # Discriminate Chars
        if escape:
            current.append(ch)
            escape = False
            continue

        if ch == "\\":
            current.append(ch)
            escape = True
            continue

        if in_string:
            current.append(ch)
            if ch == quote_char:
                in_string = False
                quote_char = None
            continue

        if ch in ['"', "'"]:
            current.append(ch)
            in_string = True
            quote_char = ch
            continue

        if ch in "([{":
            depth += 1
            current.append(ch)
            continue

        if ch in ")]}":
            depth -= 1
            current.append(ch)
            continue

        if ch == delimiter and depth == 0:
            part = "".join(current).strip()
            if part:
                parts.append(part)
            current = []
        else:
            current.append(ch)

    final = "".join(current).strip()
    if final:
        parts.append(final)

    return parts


def parse_value(value):
    """
    Converts ToolACE argument values into Python objects where possible.
    Examples:
    "AAPL" -> "AAPL"
    10 -> 10
    4.5 -> 4.5
    ["2025"] -> ["2025"]
    {"x": 1} -> {"x": 1}
    true -> True
    null -> None
    """
    value = value.strip()

    try:
        return ast.literal_eval(value)
    except Exception:
        pass

    lower = value.lower()

    if lower == "true":
        return True
    if lower == "false":
        return False
    if lower in ["none", "null"]:
        return None

    try:
        if "." in value:
            return float(value)
        return int(value)
    except Exception:
        pass

    return value


def parse_single_toolace_call(call_text):
    """
    Converts:
    SEC Filings(identifier="AAPL")

    Into:
    {
      "name": "SEC Filings",
      "arguments": {"identifier": "AAPL"}
    }
    """
    call_text = call_text.strip()

    match = re.match(r"^(.*?)\((.*)\)$", call_text, flags=re.DOTALL)

    if not match:
        return None

    tool_name = match.group(1).strip()
    args_text = match.group(2).strip()

    arguments = {}

    if args_text:
        arg_parts = split_top_level(args_text, delimiter=",")

        for arg in arg_parts:
            if "=" not in arg:
                continue

            key, raw_value = arg.split("=", 1)
            key = key.strip()
            raw_value = raw_value.strip()

            if not key:
                continue

            arguments[key] = parse_value(raw_value)

    return {
        "name": tool_name,
        "arguments": arguments
    }


def toolace_to_your_json(toolace_output, example_id="0"):
    """
    Converts ToolACE assistant output into your JSON tool_calls format.

    Input:
    [SEC Filings(identifier="AAPL")]

    Output:
    {
      "tool_calls": [
        {
          "id": "call_toolace_0_0",
          "name": "SEC Filings",
          "arguments": {"identifier": "AAPL"}
        }
      ]
    }
    """
    toolace_output = str(toolace_output or "").strip()

    if not toolace_output.startswith("[") or not toolace_output.endswith("]"):
        return None

    inner = toolace_output[1:-1].strip()

    if not inner:
        return None

    raw_calls = split_top_level(inner, delimiter=",")

    tool_calls = []

    for call_idx, raw_call in enumerate(raw_calls):
        parsed = parse_single_toolace_call(raw_call)

        if parsed is None:
            continue

        tool_calls.append({
            "id": f"call_toolace_{example_id}_{call_idx}",
            "name": parsed["name"],
            "arguments": parsed["arguments"]
        })

    if not tool_calls:
        return None

    return {
        "tool_calls": tool_calls
    }


def toolace_to_your_json_string(toolace_output, example_id="0"):
    converted = toolace_to_your_json(toolace_output, example_id=example_id)

    if converted is None:
        return None

    return json.dumps(
        converted,
        ensure_ascii=False,
        separators=(",", ":")
    )

In [1]:
from datasets import Dataset

def looks_like_tool_call(text):
    """Function to check if the Part of the dataset is a ToolACE tool call """
    text = str(text or "").strip()
    return (
        text.startswith("[")
        and text.endswith("]")
        and "(" in text
        and ")" in text
    )


def build_json_toolcall_training_examples(rows):
    """ Goes for every Row and builds the Training Dataset"""
    training_examples = []
    skipped = 0

    for row_idx, row in enumerate(rows):
        system_prompt = str(row["system"] or "").strip()
        conversations = row["conversations"]

        for msg_idx, message in enumerate(conversations):
            if message["from"] != "assistant":
                continue

            assistant_raw = str(message["value"] or "").strip()

            if not looks_like_tool_call(assistant_raw):
                continue

            # Get the JSON
            assistant_json = toolace_to_your_json_string(
                assistant_raw,
                example_id=f"{row_idx}_{msg_idx}"
            )

            if assistant_json is None:
                skipped += 1
                continue

            previous_user = None

            for back_idx in range(msg_idx - 1, -1, -1):
                if conversations[back_idx]["from"] == "user":
                    previous_user = str(conversations[back_idx]["value"] or "").strip()
                    break

            if previous_user is None:
                skipped += 1
                continue

            # Construct the Messages Array
            messages = [
                {
                    "role": "system",
                    "content": (
                        system_prompt
                        + "\n\nWhen calling tools, output only valid JSON in this exact format:\n"
                        + '{"tool_calls":[{"id":"call_x","name":"tool_name","arguments":{}}]}\n'
                        + "Do not output any extra text before or after the JSON."
                    )
                },
                {
                    "role": "user",
                    "content": previous_user
                },
                {
                    "role": "assistant",
                    "content": assistant_json
                }
            ]

            training_examples.append({
                "messages": messages,
                "raw_toolace_output": assistant_raw,
                "json_toolcall_output": assistant_json
            })

    print("Created examples:", len(training_examples))
    print("Skipped examples:", skipped)

    return training_examples


toolace_json_examples = build_json_toolcall_training_examples(toolace_5k)
toolace_json_dataset = Dataset.from_list(toolace_json_examples)

print(toolace_json_dataset)
print(toolace_json_dataset[0]["messages"])
print(toolace_json_dataset[0]["json_toolcall_output"])

NameError: name 'toolace_5k' is not defined

## Mount to google Drive for reproducability and backup of model checkpoints

- If our runtime disconnects then we can restart easy
- Much easier to donwload new weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/colab_runs/medGemmaToolAce"

In [ ]:
import os

os.makedirs(BASE_DIR, exist_ok=True)

CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
LOG_DIR = os.path.join(BASE_DIR, "logs")
MODEL_DIR = os.path.join(BASE_DIR, "final_model")

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

### Model Training
- This is where we train the Model
- Use Unsloth and TRL for PEFT Training

In [ ]:
from unsloth import FastLanguageModel

# Load MedGemma Model and the Tokenizer, to tokenize the training data
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/medgemma-4b-it",
    max_seq_length = 2048,
    dtype= None,
    load_in_4bit= True

)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [ ]:
# Applies the Template data to the Models tokenizer so Model understands the data
def apply_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


formatted_toolace_json_dataset = toolace_json_dataset.map(apply_template)

print(formatted_toolace_json_dataset[0]["text"])

Map:   0%|          | 0/4464 [00:00<?, ? examples/s]

<bos><start_of_turn>user
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out.
Here is a list of functions in JSON format that you can invoke:
[{"name": "Search Event Details", "description": "This API allows you to search for specific Mixed Martial Arts (MMA) events by fighter names. The response includes details such as the event string, fighter names, and the outcome of the event.", "parameters": {"type": "dict", "properties": {"q": {"description": "Search query for fighter names", "type": "string"}}, "required": ["q"]}, "required": null}, {"name": "Search Matchups", "description": "Search for sports matchups based on a query string.", "parameters": {"type": "dict", "properties": {"query": {"descript

In [ ]:
# Gets the PEFT model for training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing = "unsloth"

)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [ ]:
from trl import SFTTrainer, SFTConfig

# Verify Training Configuration and Parameters then start training
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_toolace_json_dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=2048,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=5e-5,
        warmup_steps=10,
        logging_steps=10,
        output_dir=CKPT_DIR,
        optim="adamw_8bit",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        packing=False,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/4464 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,464 | Num Epochs = 2 | Total steps = 1,116
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.787362
20,1.460253
30,1.235510
40,1.019876
50,0.803675
60,0.662083
70,0.633150
80,0.610257
90,0.561699
100,0.563942


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-100/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-300.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/medGemmaToolAce/checkpoints/checkpoint-

TrainOutput(global_step=1116, training_loss=0.4953841325629997, metrics={'train_runtime': 14675.6558, 'train_samples_per_second': 0.608, 'train_steps_per_second': 0.076, 'total_flos': 1.9018025733435456e+17, 'train_loss': 0.4953841325629997, 'epoch': 2.0})

In [ ]:
from unsloth import FastLanguageModel
import torch

CHECKPOINT_PATH = "/content/drive/MyDrive/colab_runs/es1_agent_v2/checkpoints/checkpoint-752"
max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CHECKPOINT_PATH,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Gemma3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): SiglipVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
              (position_embedding): Embedding(4096, 1152)
            )
            (encoder): SiglipEncoder(
              (layers): ModuleList(
                (0-15): 16 x SiglipEncoderLayer(
                  (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                  (self_attn): SiglipAttention(
                    (k_proj): lora.Linear(
                      (base_layer): Linear(in_features=1152, out_features=1152, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.05, inplace=False)
                      )
                      (lor

In [ ]:
prompt = """### Instruction:
Analyze the patient’s condition and assess if they require immediate life-saving interventions. Focus on identifying critical needs in airway, breathing, circulation, or severe physiological instability. Provide specific reasons for your assessment.

### Input:
A 70-year-old male with stage IV pancreatic cancer presents with dyspnea, dizziness, diarrhea, hypotension (BP 86/57), pulse 145, RR 30, SpO2 100%, poor oral intake, and dehydration.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        use_cache=True,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Analyze the patient’s condition and assess if they require immediate life-saving interventions. Focus on identifying critical needs in airway, breathing, circulation, or severe physiological instability. Provide specific reasons for your assessment.

### Input:
A 70-year-old male with stage IV pancreatic cancer presents with dyspnea, dizziness, diarrhea, hypotension (BP 86/57), pulse 145, RR 30, SpO2 100%, poor oral intake, and dehydration.

### Response:
The patient is a 70-year-old male with stage IV pancreatic cancer, presenting with several concerning symptoms that suggest he may be experiencing a critical condition. The symptoms include dyspnea (difficulty breathing), dizziness, diarrhea, hypotension (low blood pressure), tachycardia (high heart rate), tachypnea (high respiratory rate), and poor oral intake with dehydration.

### Analysis:
1. **Hypotension and Tachycardia**: The combination of low blood pressure and high heart rate indicates possible shock, which 